## Note
This notebook is not intended to be re-run because it depends on:
- a private API key stored in a local .env file
- external API calls to Mapbox


In [ ]:
import json
import os
import re
import time
import geopandas as gpd
import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import clear_output, display

load_dotenv()

MAPBOX_TOKEN = os.getenv("MAPBOX_TOKEN")
BATCH_URL = (
    f"https://api.mapbox.com/search/geocode/v6/batch?access_token={MAPBOX_TOKEN}"
    if MAPBOX_TOKEN
    else None
)

CHECKPOINT_FILE = "geocoded_irish_properties_train_checkpoint.csv"
BATCH_SIZE = 1000

In [ ]:
# Example with train data
raw_df = pd.read_csv('../data/ppr-group-25200353-train-clean-final.csv')

print(f"loaded {len(raw_df)} records")

raw_df['mapbox_query'] = raw_df.apply(
    axis=1, 
    address_col='address',
    county_col='county'
)

raw_df.head()

In [ ]:
if os.path.exists(CHECKPOINT_FILE):
    df = pd.read_csv(CHECKPOINT_FILE, low_memory=False)
    processed_count = df['confidence'].notna().sum()
    print(f"loaded checkpoint. {processed_count} addresses already processed")
else:
    df = raw_df.copy()
    df['latitude'] = pd.NA
    df['longitude'] = pd.NA
    df['confidence'] = pd.NA
    df['feature_type'] = pd.NA
    df['match_code'] = pd.NA
    df.to_csv(CHECKPOINT_FILE, index=False)

df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

In [ ]:
def run_geocoding_loop(df):
    while True:
        # Treat rows with missing coordinates as unprocessed
        unprocessed = df[df['latitude'].isna() | df['longitude'].isna()]

        if unprocessed.empty:
            print("all addresses processed")
            break

        batch_df = unprocessed.head(BATCH_SIZE).copy()

        # Keep only rows with a real query
        batch_df = batch_df[batch_df['mapbox_query'].notna()]

        if batch_df.empty:
            print("no valid mapbox queries left to process")
            break

        indices = batch_df.index.tolist()

        payload = [
            {"q": str(query), "country": "ie", "limit": 1}
            for query in batch_df['mapbox_query']
        ]

        print(f"sending {len(payload)} addresses...")

        response = requests.post(BATCH_URL, json=payload)

        if response.status_code != 200:
            print(f"API Error {response.status_code}: {response.text}")
            break

        data = response.json()
        results_array = data.get('batch', data) if isinstance(data, dict) else data

        needs_review = False

        for i, res in enumerate(results_array):
            df_idx = indices[i]
            features = res.get('features', [])

            if features:
                feature = features[0]
                coords = feature['geometry']['coordinates']
                props = feature.get('properties', {})

                feature_type = props.get('feature_type', 'unknown')
                match_code_dict = props.get('match_code', {})

                conf = props.get('confidence')
                if not conf:
                    conf = match_code_dict.get('confidence', 'none')

                df.loc[df_idx, 'longitude'] = coords[0]
                df.loc[df_idx, 'latitude'] = coords[1]
                df.loc[df_idx, 'feature_type'] = feature_type
                df.loc[df_idx, 'confidence'] = conf
                df.loc[df_idx, 'match_code'] = json.dumps(match_code_dict)

            else:
                df.loc[df_idx, 'confidence'] = 'not_found'
                df.loc[df_idx, 'feature_type'] = 'not_found'
                df.loc[df_idx, 'match_code'] = json.dumps({"not_found": True})
                needs_review = True

        df.to_csv(CHECKPOINT_FILE, index=False)

        if needs_review:
            print("mapbox cannot find at least one address. review in next cell")
            break
        else:
            print("moving to next...")

run_geocoding_loop(df)

df['latitude'] = pd.to_numeric(df.get('latitude'), errors='coerce')
df['longitude'] = pd.to_numeric(df.get('longitude'), errors='coerce')

df_valid = df.dropna(subset=['latitude', 'longitude']).copy()
print(df_valid.shape)
print(df[['latitude', 'longitude']].isna().sum())

In [ ]:
from IPython.display import clear_output

failed_mask = df['confidence'] == 'not_found'
failed_indices = df[failed_mask].index

if len(failed_indices) == 0:
    print("no failed addresses to review")
else:
    print(f"found {len(failed_indices)} addresses that need manual entry\n")
    
    for idx in failed_indices:
        raw_address = df.loc[idx, 'address']
        query = df.loc[idx, 'mapbox_query']
        
        print(f"Row {idx} | Raw Address: {raw_address}")
        print(f"Mapbox Query: {query}")
        
        lat_input = input("Enter Latitude (or press Enter to skip & leave blank): ").strip()
            
        lon_input = input("Enter Longitude: ").strip()
        
        try:
            df.loc[idx, 'latitude'] = float(lat_input)
            df.loc[idx, 'longitude'] = float(lon_input)
            df.loc[idx, 'confidence'] = 'manual'
            df.loc[idx, 'feature_type'] = 'manual'
            df.loc[idx, 'match_code'] = '{"manual_override": true}'
            
            clear_output(wait=True)
            print(f"saved manual coordinates for row {idx}\n")
            
        except ValueError:
            clear_output(wait=True)
            print(f"invalid coordinate format. Skipped row {idx}\n")

    # Save manual work immediately
    df.to_csv(CHECKPOINT_FILE, index=False)
    print("row saved")

## Categorizing into Local Electoral and Small Areas

In [ ]:
import geopandas as gpd

df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

lea_columns_to_keep = ['CSO_LEA', 'geometry']
gdf_lea = gpd.read_file('../data/cso_lea.geojson')[lea_columns_to_keep]

df_valid = df.dropna(subset=['latitude', 'longitude']).copy()

print(df[['latitude', 'longitude']].isna().sum())
print(df[['latitude', 'longitude']].head())

properties = gpd.GeoDataFrame(
  df_valid,
  geometry=gpd.points_from_xy(df_valid['longitude'], df_valid['latitude']),
  crs='EPSG:4326'
)

properties = properties.to_crs(gdf_lea.crs)

joined_data = gpd.sjoin(
  properties,
  gdf_lea,
  how='left',
  predicate='within'
)

joined_data = joined_data.rename(columns={
  'CSO_LEA': 'cso_lea',
})

In [ ]:
small_area_columns_to_keep = ['SA_PUB2022', 'geometry']
gdf_small_area = gpd.read_file('../data/cso_small_areas.geojson')[small_area_columns_to_keep]

joined_data = joined_data.drop(columns=['index_right'], errors='ignore')

properties_small_area = gpd.GeoDataFrame(
  joined_data,
  geometry=gpd.points_from_xy(joined_data['longitude'], joined_data['latitude']),
  crs='EPSG:4326'
)

properties_small_area = properties_small_area.to_crs(gdf_small_area.crs)

joined_data = gpd.sjoin(
  properties_small_area,
  gdf_small_area,
  how='left',
  predicate='within'
)

joined_data = joined_data.rename(columns={
  'SA_PUB2022': 'cso_small_area',
})

joined_data = joined_data.drop(columns=['geometry', 'index_right'], errors='ignore')
joined_data.head()

In [ ]:
joined_data.to_csv('../data/ppr-group-25200353-train-geocoded.csv', index=False)